# Integration of all plates into a single mouse dataset


In [ ]:
# Loading packages from omicverse and scanpy (Python)
import scanpy as sc
import os
import numpy as np
from datetime import date

In [ ]:
mouse = "PC118"
file_path = f"F:\Data\snRNAseq\PlaceCell\FilteredData\{mouse}"

batches = {
    sid: os.path.join(file_path, f"{sid}_counts_martix_QC.h5ad")
    for sid in [f"{mouse}-1", f"{mouse}-2", f"{mouse}-3"]
}

adatas = {}

for sid, path in batches.items():
    print(f"🔹 Reading batches {sid} from: {path}")
    ad = sc.read_h5ad(path)
    ad.obs["batches"] = sid
    ad.var_names_make_unique()
    adatas[sid] = ad

In [ ]:
adata = sc.concat(adatas, label="batches", join="outer")
adata

In [ ]:
adata.X[np.isnan(adata.X)] = 0

In [ ]:
nan_count = np.isnan(adata.X).sum()
print(f"Number of NaN values in adata.X: {nan_count}")
print(f"Max value in martix:" ,adata.X.max())

In [ ]:
# import re

# # 使用 apply() 来修改 obs_names
# adata.obs_names = adata.obs_names.to_series().apply(lambda x: re.sub(r'(-SS3)+', '-SS3', x))

# # 打印结果查看修改后的值
# print(adata.obs_names)

In [ ]:
adata.obs[["batches"]].drop_duplicates()

In [ ]:
adata.obs_names_make_unique()
print(adata.obs["batches"].value_counts())
adata

In [ ]:
# adata.obs["condition"] = "nonplace"
adata.obs.loc[adata.obs["property"].isin([830]), "condition"] = "silent"
adata.obs.loc[adata.obs["property"].isin([750]), "condition"] = "place"

In [ ]:
obs_to_keep = ["well","property","condition","batches"]
adata.obs = adata.obs[obs_to_keep]
adata

In [ ]:
adata.obs

In [ ]:
adata.layers["counts"] = adata.X.copy()  

In [ ]:
# 检查是否为整数矩阵（允许 float dtype）
is_int = np.allclose(adata.X.data, np.round(adata.X.data))
print("X looks like integer UMI matrix:", is_int)

print("Max in X:", np.max(adata.X))
print("Max in layers['counts']:", np.max(adata.layers['counts']))


In [ ]:
today = date.today().strftime("%Y%m%d")
from datetime import date
output_path = os.path.join(file_path, f"{mouse}_count_martix_QC_{today}.h5ad")

adata.write_h5ad(output_path)
print(f"Successful saving：{output_path}")